In [1]:
!git clone https://github.com/GonnaMakeYouCry/HSE-Agent-Systems_2026.git
%cd HSE-Agent-Systems_2026

Cloning into 'HSE-Agent-Systems_2026'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 22 (delta 2), reused 20 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 208.45 KiB | 2.19 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/HSE-Agent-Systems_2026


In [2]:
!pip install -e


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

-e option requires 1 argument


In [5]:
!pip install openai tenacity pandas tqdm

# 🏠 Домашнее задание №1 — Решаем математику

## Цель

Выбить 100\100 по датасету

## Описание

Вам дан датасет с математическими вопросами (уравнения, арифметика, сравнения и др.).  
Каждый вопрос имеет:
- `question_id` — уникальный идентификатор
- `question` — текст задачи  
- `answer` — правильный ответ


**Ваша задача:**
1. Решить задачи с помощью LLM
2. Написать LLM flow, который для каждого вопроса:
   - отправляет вопрос модели
   - получаете результат
Решение может быть с помощью SGR, function_call. Решайте как считаете правильным. Главное результат

4. Прогнать на всём датасете и получить итоговый **score**

## Критерии оценки

- **Accuracy** — доля правильных ответов (совпадение значения)

---

> 💡 Вы можете использовать **любого провайдера** и **любую модель** с поддержкой

---

## 📊 Оценка ДЗ (10-балльная шкала)

| Критерий | Баллы |
|---|---|
| **Промпт** (применены минимум 2 техники из лекции) | **1** |
| **Accuracy** ниже 85% | **0** |
| **Accuracy** 85–95% | **3** |
| **Accuracy** 95–99% | **4** |
| **Accuracy** 100% | **5** |
| Применение **SGR** и/или **function call** | **+2** |
| Дешёвая модель (< 20 руб / 1M токенов input И output) | **+2** |



#### Примеры дешёвых моделей: `Ministral 3 14B 2512`, `gpt-oss-20b`, `Qwen2.5-VL 7B Instruct`


### ⚠️ Штрафы

| Нарушение | Штраф |
|---|---|
| Модель дороже 200 руб / 1M токенов (input) | **−2** |
| Описание проблемных кейсов в промпте | **−2** |
| Нет retry в случае ошибки при запросе по API | **−1** |


#### "Описание проблемных кейсов в промпте" Это например указать в промпте что для вопроса 'Solve -34*v + 232*v + 52351 = 48985 for v.	' ответ -17

#### За модели средней стоимости (20–200 руб / 1M токенов) штрафов нет.

## 0. Настройка окружения

In [10]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
from google.colab import userdata
import os

os.environ["POLZA_AI_API_KEY"] = userdata.get("POLZA_AI_API_KEY")
os.environ["GIGACHAT_API_KEY"] = "key_for_validation"

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.token_manager import GigaTokenManager
from src.config import settings
from src.utils import Answer, validate_answers

token_manager = GigaTokenManager()

## 1. Загрузка датасета

Датасет содержит математические вопросы разных типов. Загрузим его и посмотрим на структуру.

In [11]:
import pandas as pd
dataset = pd.read_excel(PROJECT_ROOT / "lectures" / "lecture_1" / "hw_1_dataset.xlsx")

In [12]:
dataset

,question_id,question,answer
0,4d637ecc-d8a8-4ef5-9467-259f51ebe692,Solve -282*d + 929 - 178 = -1223 for d.,7
1,27534870-1f8c-41e7-91c8-61a815c00392,Solve 49*l + 45*l - 125 - 63 = 0 for l.,2
2,cb319da5-2d1e-4814-a665-686521c56c02,Solve -64*t + 1387 - 848 + 933 = 0 for t.,23
3,0f9540f4-91cb-4d58-9e91-93931e4a8aeb,Solve 75*g = 192*g - 71*g - 79*g - 264 for g.,-8
4,d05be1f2-eb77-4175-915f-68c3269983ab,Solve -34*v + 232*v + 52351 = 48985 for v.,-17
...,...,...,...
106,d3bbfb3c-1674-4d6c-b883-21e699417455,"Sort 53, 279, 2, 1, 5, 4, -5.","-5, 1, 2, 4, 5, 53, 279"
107,f4c87be8-280e-4f46-90f6-4468641ec58e,"Sort 2, -0.2, -198, -1860.","-1860, -198, -0.2, 2"
108,1585d0d6-0fa0-46b9-9c05-3e91d96994ef,"Sort 5, -175, -4, -7, -30.","-175, -30, -7, -4, 5"
109,36033990-219d-4065-a2a5-08fa1cb8e7e4,"Sort -2, 3936, 15/19 in ascending order.","-2, 15/19, 3936"


In [9]:
print(f"Всего вопросов: {len(dataset)}")

Всего вопросов: 111


## 2. Настройка LLM-клиента

Создайте клиент для работы с LLM. Можно использовать любого провайдера

> ⚠️ Убедитесь, что выбранная модель поддерживает **function calling** (Если будете использовать function calling).

In [22]:
from openai import OpenAI

# =====================================================================
# TODO: Настройте клиент и модель
# =====================================================================

client = OpenAI(
    api_key=os.environ["POLZA_AI_API_KEY"],
    base_url="https://polza.ai/api/v1"
)

model = "qwen/qwen-2.5-7b-instruct"

## 3. Обработка одного вопроса

Реализуйте функцию `ask_question()`, которая:

1. Отправляет вопрос модели
2. Проверяет, вернула ли модель `function_call` (если решаете использовать `function_call`)
3. Если да — парсит аргументы, вызывает нужный function, возвращает результат модели
4. Получает финальный ответ


In [26]:
import json
import time

# Дополните промпт при необходимости
SYSTEM_PROMPT = (
    "Ты элитный математик-вычислитель. Твоя задача — решить математическую задачу "
    "или задачу сортировки с абсолютной точностью.\n\n"
    "Используй строгий формат JSON с двумя полями:\n"
    "1. 'reasoning': здесь напиши пошаговое логическое решение задачи.\n"
    "2. 'answer': здесь напиши ТОЛЬКО финальный результат (например '7', '-17', '-2, 15/19, 3936').\n\n"
    "Возвращай только валидный JSON."
)



def ask_question(question: str) -> str | None:
    """
    Отправляет один вопрос LLM-агенту и возвращает финальный текстовый ответ.
    Использует SGR (JSON mode) и паттерн Retry для обработки ошибок сети.
    """
    max_retries = 3

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": question},
                ],
                response_format={"type": "json_object",},
                temperature=0.01,
            )

            msg = resp.choices[0].message
            parsed_data = json.loads(msg.content)
            final_answer = parsed_data.get("answer")

            if final_answer is not None:
                return str(final_answer).strip()
            else:
                raise ValueError("Ответ не содержит ожидаемого поля 'answer'")

        except Exception as e:
            print(f"Ошибка: {e}, на попытке {attempt + 1}")
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                return None

    return None


### Проверка на одном вопросе

Прежде чем запускать на всём датасете, проверьте агента на одном примере.

In [ ]:
# Тест на одном примере
test_row = dataset.iloc[0]
print(f"Вопрос:   {test_row['question']}")
print(f"Ожидаемо: {test_row['answer']}")

answer = ask_question(test_row["question"])
print(f"Ответ LLM: {answer}")

if answer is not None:
    print(f"Совпадает: {answer == test_row['answer']}")

## 4. Прогон по всему датасету

Запустите агента на всех вопросах из датасета. Для каждого вопроса:
1. Вызовите `ask_question()` для получения ответа
2. Приведите ответ к нужному типу через `parse_answer()`
3. Сформируйте объект `Answer(question_id=..., answer=...)`

> ⏱️ Этот шаг может занять время — в зависимости от провайдера и количества вопросов.  
> При необходимости добавьте `time.sleep()` между запросами для соблюдения rate limits.

In [ ]:
from tqdm.auto import tqdm

answers: list[Answer] = []
failed: list[dict] = []

for idx, row in tqdm(dataset.iterrows(), total=len(dataset), desc="Обработка вопросов"):
    question_id = row["question_id"]
    question = row["question"]

    # Получаем ответ от агента
    answer = ask_question(question)

    if answer is None:
        failed.append({"index": idx, "question_id": question_id, "reason": "no_answer"})
        continue

    try:
        answers.append(Answer(question_id=question_id, answer=answer))
    except Exception as e:
        failed.append({
            "index": idx,
            "question_id": question_id,
            "reason": f"parse_error: {e}",
            "answer": answer,
        })

    # Пауза между запросами (при необходимости)
    time.sleep(0.5)

print(f"\nГотово!")
print(f"Ответов получено: {len(answers)} из {len(dataset)}")
print(f"Ошибок: {len(failed)}")

Обработка вопросов:   0%|          | 0/111 [00:00<?, ?it/s]

Ошибка: Expecting value: line 1 column 1 (char 0), на попытке 1
Ошибка: Expecting value: line 1 column 1 (char 0), на попытке 2
Ошибка: Expecting value: line 1 column 1 (char 0), на попытке 3


## 5. Валидация и подсчёт score

Используем функцию `validate_answers()` из `src.utils` для проверки ответов.  
Валидация проверяет:
- **Значение** — ответ должен **точно совпадать** с эталоном (сравнение по строковому представлению)

### Формула score

$$\text{accuracy} = \frac{\text{правильных ответов}}{\text{всего вопросов в датасете}}$$

In [ ]:
# Валидация ответов
result = validate_answers(answers, dataset)

total_questions = len(dataset)
total_answered = len(answers)
total_correct = total_answered - len(result["errors"])

# Score считается от общего числа вопросов в датасете
accuracy = total_correct / total_questions if total_questions > 0 else 0

print("=" * 50)
print(f"📊 РЕЗУЛЬТАТЫ")
print("=" * 50)
print(f"Всего вопросов в датасете: {total_questions}")
print(f"Ответов отправлено:        {total_answered}")
print(f"Не удалось ответить:       {len(failed)}")
print(f"Ошибок валидации:          {len(result['errors'])}")
print(f"Правильных ответов:        {total_correct}")
print(f"")
print(f"🎯 Accuracy: {accuracy:.2%} ({total_correct}/{total_questions})")
print("=" * 50)

In [ ]:
# Детализация ошибок (первые 10)
if result["errors"]:
    print("Примеры ошибок:")
    for err in result["errors"][:10]:
        qid = err["question_id"]
        reason = err["reason"]
        row = dataset[dataset["question_id"] == qid]
        if not row.empty:
            q = row.iloc[0]["question"]
            print(f"\n  [{reason}] {q[:80]}...")
            if reason == "value_mismatch":
                print(f"    Ожидалось: {err['expected']}")
                print(f"    Получено:  {err['actual']}")
else:
    print("✅ Все ответы корректны!")

In [ ]:
# Accuracy по типам ответов
error_qids = {e["question_id"] for e in result["errors"]}
failed_qids = {f["question_id"] for f in failed}

print("Accuracy по типам ответов:")
print("-" * 40)
for atype in ["int", "float", "bool", "list", "str"]:
    subset = dataset[dataset["answer_type"] == atype]
    total = len(subset)
    wrong = sum(1 for _, r in subset.iterrows()
                if str(r["question_id"]) in error_qids or str(r["question_id"]) in failed_qids)
    correct = total - wrong
    acc = correct / total if total > 0 else 0
    print(f"  {atype:>5}: {acc:.0%}  ({correct}/{total})")

## 📝 Что нужно сдать

1. Этот ноутбук с **заполненным кодом** (все `TODO` реализованы)
2. Все ячейки должны быть **выполнены** — с выводом результатов
3. В выводе должен быть виден **итоговый score**

---

## 💡 Советы

- **Начните с малого**: сначала отладьте работу на единичных запросах или на малом датасете
- **Промпт имеет значение**: хороший `SYSTEM_PROMPT` сильно влияет на качество ответов
- **Обрабатывайте ошибки**: LLM иногда возвращает неожиданные ответы — ваш код не должен падать
---

GLHF 🚀